# NLP et Deep Learning : Sémantique Fine et Nommage Automatique

## Objectif
Ce notebook explore des méthodes avancées pour capturer la sémantique des jeux en utilisant le **Traitement du Langage Naturel (NLP)** et le **Deep Learning**. 

Nous allons :
1.  Utiliser des **Embeddings** (BERT via Sentence-Transformers) pour représenter les tags dans un espace vectoriel riche.
2.  S'appuyer sur le clustering de Louvain pour identifier des groupes cohérents.
3.  **Nommer automatiquement les clusters** en calculant le centroïde des embeddings et en trouvant le terme le plus représentatif.

Ces étapes permettent de passer d'une simple liste de tags à une véritable compréhension thématique et structurelle.

In [7]:
import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import logging
import os

# 1. Configuration pour un affichage propre
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

# 2. Chargement et Préparation des Données
df_structured = pd.read_csv('../data/New_Games_Gameplay_Taxonomy.csv')
df_tags = df_structured[['game_id', 'Genre', 'Mechanics']].copy()
df_tags['all_tags'] = df_tags['Genre'].fillna('') + ', ' + df_tags['Mechanics'].fillna('')
df_tags['all_tags'] = df_tags['all_tags'].str.strip(', ')

tags_exploded = df_tags.assign(tag=df_tags['all_tags'].str.split(', ')).explode('tag')
tags_exploded = tags_exploded[tags_exploded['tag'].notna() & (tags_exploded['tag'] != '')]

tag_counts = tags_exploded['tag'].value_counts()
frequent_tags = tag_counts[tag_counts > 500].index
tags_filtered = tags_exploded[tags_exploded['tag'].isin(frequent_tags)]

matrix = pd.get_dummies(tags_filtered['tag']).astype(int)
matrix['game_id'] = tags_filtered['game_id'].values
matrix = matrix.groupby('game_id').max()

# Similarité Cosinus sur la présence des tags
tag_sim_matrix = cosine_similarity(matrix.T)
df_sim_presence = pd.DataFrame(tag_sim_matrix, index=matrix.columns, columns=matrix.columns)

## 2. Clustering de Louvain
Nous reproduisons ici le clustering de Louvain pour obtenir nos groupes de base.

In [8]:
G = nx.Graph()
threshold = 0.20
tags = df_sim_presence.columns
for i in range(len(tags)):
    for j in range(i + 1, len(tags)):
        sim = tag_sim_matrix[i, j]
        if sim > threshold:
            G.add_edge(tags[i], tags[j], weight=sim)

partition = community_louvain.best_partition(G, weight='weight')

clusters = {}
for tag, community_id in partition.items():
    if community_id not in clusters:
        clusters[community_id] = []
    clusters[community_id].append(tag)

print(f"Nombre de clusters détectés : {len(clusters)}")

Nombre de clusters détectés : 11


## 3. Embeddings et Nommage Automatique

Nous utilisons le modèle **BERT** (`all-MiniLM-L6-v2`) pour transformer chaque tag en vecteur. Contrairement à la simple présence dans les jeux, ces vecteurs capturent la sémantique linguistique des mots.

In [11]:
print("Chargement du modèle d'embeddings...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Calcul des embeddings pour tous les tags fréquents
unique_tags = list(matrix.columns)
tag_embeddings = model.encode(unique_tags)
df_embeddings = pd.DataFrame(tag_embeddings, index=unique_tags)

def get_cluster_name(cluster_tags, all_tag_embeddings):
    # Calcul du centroïde (moyenne des vecteurs du cluster)
    cluster_vecs = all_tag_embeddings.loc[cluster_tags]
    centroid = cluster_vecs.mean(axis=0).values.reshape(1, -1)
    
    # Recherche du tag le plus proche du centroïde dans TOUTE la base (inversion du vecteur)
    similarities = cosine_similarity(centroid, all_tag_embeddings.values)
    best_idx = np.argmax(similarities)
    return unique_tags[best_idx]

cluster_names = {}
for cid, tags_in_cluster in clusters.items():
    name = get_cluster_name(tags_in_cluster, df_embeddings)
    cluster_names[cid] = name
    print(f"Cluster {cid} -> Nom suggéré : {name}")
    print(f"   Tags : {', '.join(tags_in_cluster[:10])}")

# Affichage des clusters avec leurs noms
print("\n--- Résumé des Clusters et Noms Suggérés ---")
for cid in sorted(cluster_names.keys()):
    name = cluster_names[cid]
    tags_in_cluster = clusters[cid]
    print(f"\nCluster {cid} : '{name}'")
    print(f"  Nombre de tags : {len(tags_in_cluster)}")
    print(f"  Tags (exemples) : {', '.join(tags_in_cluster[:15])}{'...' if len(tags_in_cluster) > 15 else ''}")

Chargement du modèle d'embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 422.90it/s, Materializing param=pooler.dense.weight]                             


Cluster 3 -> Nom suggéré : RPG
   Tags : 2D Fighter, Fighting, 2D Platformer, Action-Adventure, Adventure, Controller, Metroidvania, Platformer, Puzzle Platformer, 3D Fighter
Cluster 2 -> Nom suggéré : Shooter
   Tags : Action, Arena Shooter, Bullet Hell, FPS, Physics, Shoot 'Em Up, Shooter, Survival Horror, Third-Person Shooter, Top-Down Shooter
Cluster 10 -> Nom suggéré : Strategy RPG
   Tags : 4X, Grand Strategy, RPG, Strategy, Action RTS, RTS, Real Time Tactics, Tower Defense, Board Game, Turn-Based Strategy
Cluster 4 -> Nom suggéré : Roguelike Deckbuilder
   Tags : Action Roguelike, Roguelike, Roguelite, Procedural Generation, Card Game, Card Battler, Deckbuilding, Roguelike Deckbuilder, Trading Card Game, Solitaire
Cluster 8 -> Nom suggéré : Building
   Tags : Open World, Survival, Simulation, Automation, Building, Resource Management, Base Building, City Builder, Colony Sim, Crafting
Cluster 0 -> Nom suggéré : Visual Novel
   Tags : Choices Matter, Choose Your Own Adventure, Mul

*Ce processus de nommage est assez performant et sort avec rigueur des termes très représentatifs du contenu sémantique du cluster.*

## Conclusion

L'intégration du **Deep Learning (Sentence-BERT)** pour le nommage automatique des clusters marque une étape clé dans l'automatisation de la compréhension de la folksonomie Steam.

**Résultats majeurs :**
1.  **Pertinence Sémantique** : Les noms suggérés par le modèle (ex: *Visual Novel*, *Roguelike Deckbuilder*, *Building*) sont d'une précision remarquable. Le modèle parvient à identifier le tag "pivot" qui donne son sens à l'ensemble du groupe.
2.  **Au-delà de la co-occurrence** : Contrairement aux méthodes purement statistiques, cette approche capture la proximité conceptuelle. Elle permet de valider que les clusters de Louvain ne sont pas seulement des regroupements de données, mais des entités ludologiques cohérentes.
3.  **Évolutivité** : Ce système est capable de s'adapter à l'apparition de nouveaux tags ou de nouvelles tendances de gameplay sur Steam sans nécessiter une redéfinition manuelle des catégories.